# Accepted Loan Cleaning: LendingClub 2007-2018Q4

## Purpose

This notebook turns the accepted-loan EDA decisions into a repeatable cleaning workflow for supervised default-risk modeling.

It does **not** clean rejected applications. Rejected applications have no observed repayment outcome and belong in a separate population-comparison or reject-inference workflow.

## Cleaning Principles

- Preserve raw source data; write treated outputs separately.
- Use accepted/originated loans only.
- Build `target_bad` from strict baseline training statuses only: `Fully Paid` and `Charged Off`.
- Remove identifiers, target fields, and post-origination leakage before creating the model matrix.
- Prefer reviewed cleaned helper fields over raw strings, date fields, or FICO bounds.
- Use stratified sampling for local cleaning QA so charged-off outcomes are represented.
- Fit imputation, scaling, and encoding later inside the train-only modeling pipeline.


## 1. Configuration

**Description:** Define source data, output folders, sampling controls, and execution switches.

**Importance:** Cleaning should be reproducible and should never overwrite the raw LendingClub file.

**Process Made:** The notebook discovers the accepted-loan file, creates cleaning output folders, and exposes one switch for sample versus full cleaning.

**Expected Results:** Valid paths and writable outputs under `Cleaning/cleaning_outputs/`.

**Caveats:** `RUN_FULL_CLEANING = False` is the safer default for development. Set it to `True` only when ready to clean all eligible rows.


In [ ]:
from __future__ import annotations

import gzip
import os
import platform
import random
import re
from dataclasses import dataclass
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)

RANDOM_STATE = 42
DEFAULT_PROJECT_ROOT = Path(
    "/Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/"
    "Final_Project/Final/CreditRiskRAG"
)


def find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "EDA").exists() and (candidate / "Cleaning").exists() and (candidate / "README.md").exists():
            return candidate
    return DEFAULT_PROJECT_ROOT


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT.parent / "Data" / "archive"
OUTPUT_ROOT = PROJECT_ROOT / "Cleaning" / "cleaning_outputs"
TABLE_DIR = OUTPUT_ROOT / "tables"
DATASET_DIR = OUTPUT_ROOT / "datasets"
TABLE_DIR.mkdir(parents=True, exist_ok=True)
DATASET_DIR.mkdir(parents=True, exist_ok=True)

RUN_FULL_CLEANING = False
CLEANING_SAMPLE_ROWS = 250_000
MIN_ROWS_PER_STATUS_IN_SAMPLE = 1_000
CHUNK_SIZE = 250_000

ACCEPTED_PATH = DATA_ROOT / "accepted_2007_to_2018Q4.csv.gz"
if not ACCEPTED_PATH.exists():
    raise FileNotFoundError(f"Missing accepted-loan source file: {ACCEPTED_PATH}")

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("PROJECT_ROOT:", PROJECT_ROOT)
print("Accepted source:", ACCEPTED_PATH)
print("Output root:", OUTPUT_ROOT)
print("RUN_FULL_CLEANING:", RUN_FULL_CLEANING)


## 2. Feature Registry And Removal Controls

**Description:** Define the fields used for cleaning, target creation, traceability, leakage exclusion, and first-pass modeling.

**Importance:** Credit-risk cleaning should be explicit. Automatic dtype inference or broad column inclusion can accidentally keep identifiers, outcomes, or post-origination leakage.

**Process Made:** The notebook reuses the EDA-reviewed groups: application-time candidates, identifiers, target fields, leakage-demo fields, broader leakage fields, and the 38-feature starter model set.

**Expected Results:** A transparent registry that separates raw fields used for treatment from fields allowed in the final model matrix.

**Caveats:** A field being cleaned does not mean it is approved for modeling. Cleaning is a prerequisite; feature approval still depends on timing, leakage, missingness, fair-lending, and stability review.


In [ ]:
APPLICATION_TIME_CANDIDATES = [
    "loan_amnt", "term", "int_rate", "installment", "grade", "sub_grade",
    "emp_length", "home_ownership", "annual_inc", "verification_status",
    "issue_d", "purpose", "zip_code", "addr_state", "dti", "delinq_2yrs",
    "earliest_cr_line", "fico_range_low", "fico_range_high", "inq_last_6mths",
    "mths_since_last_delinq", "mths_since_last_record", "open_acc", "pub_rec",
    "revol_bal", "revol_util", "total_acc", "initial_list_status", "collections_12_mths_ex_med",
    "mths_since_last_major_derog", "policy_code", "application_type", "annual_inc_joint",
    "dti_joint", "verification_status_joint", "acc_now_delinq", "tot_coll_amt", "tot_cur_bal",
    "open_acc_6m", "open_act_il", "open_il_12m", "open_il_24m", "mths_since_rcnt_il",
    "total_bal_il", "il_util", "open_rv_12m", "open_rv_24m", "max_bal_bc", "all_util",
    "total_rev_hi_lim", "inq_fi", "total_cu_tl", "inq_last_12m", "acc_open_past_24mths",
    "avg_cur_bal", "bc_open_to_buy", "bc_util", "chargeoff_within_12_mths", "delinq_amnt",
    "mo_sin_old_il_acct", "mo_sin_old_rev_tl_op", "mo_sin_rcnt_rev_tl_op", "mo_sin_rcnt_tl",
    "mort_acc", "mths_since_recent_bc", "mths_since_recent_bc_dlq", "mths_since_recent_inq",
    "mths_since_recent_revol_delinq", "num_accts_ever_120_pd", "num_actv_bc_tl",
    "num_actv_rev_tl", "num_bc_sats", "num_bc_tl", "num_il_tl", "num_op_rev_tl",
    "num_rev_accts", "num_rev_tl_bal_gt_0", "num_sats", "num_tl_120dpd_2m", "num_tl_30dpd",
    "num_tl_90g_dpd_24m", "num_tl_op_past_12m", "pct_tl_nvr_dlq", "percent_bc_gt_75",
    "pub_rec_bankruptcies", "tax_liens", "tot_hi_cred_lim", "total_bal_ex_mort", "total_bc_limit",
    "total_il_high_credit_limit", "disbursement_method",
]

IDENTIFIER_COLS = ["id", "member_id", "url"]
TARGET_COLS = ["loan_status"]
LEAKAGE_DEMO_COLS = [
    "out_prncp", "total_pymnt", "total_rec_prncp", "total_rec_int", "recoveries",
    "collection_recovery_fee", "last_pymnt_d", "last_pymnt_amnt", "last_credit_pull_d",
    "last_fico_range_high", "last_fico_range_low", "debt_settlement_flag", "hardship_flag",
]

CLEAR_LEAKAGE_EXCLUDE = [
    "pymnt_plan", "out_prncp", "out_prncp_inv", "total_pymnt", "total_pymnt_inv",
    "total_rec_prncp", "total_rec_int", "total_rec_late_fee", "recoveries",
    "collection_recovery_fee", "last_pymnt_d", "last_pymnt_amnt", "next_pymnt_d",
    "last_credit_pull_d", "last_fico_range_high", "last_fico_range_low", "hardship_flag",
    "hardship_type", "hardship_reason", "hardship_status", "deferral_term", "hardship_amount",
    "hardship_start_date", "hardship_end_date", "payment_plan_start_date", "hardship_length",
    "hardship_dpd", "hardship_loan_status", "orig_projected_additional_accrued_interest",
    "hardship_payoff_balance_amount", "hardship_last_payment_amount", "debt_settlement_flag",
    "debt_settlement_flag_date", "settlement_status", "settlement_date", "settlement_amount",
    "settlement_percentage", "settlement_term",
]

REVIEW_BEFORE_MODELING = [
    "funded_amnt", "funded_amnt_inv", "url", "desc", "title", "zip_code",
    "initial_list_status", "policy_code", "disbursement_method", "emp_title", "addr_state",
]

SAFE_STARTER_FEATURES = [
    "loan_amnt", "term_months", "int_rate_clean", "installment", "grade", "sub_grade",
    "emp_length_years", "home_ownership", "annual_inc", "verification_status", "purpose",
    "dti", "delinq_2yrs", "fico_mean", "inq_last_6mths", "mths_since_last_delinq",
    "mths_since_last_record", "open_acc", "pub_rec", "revol_bal", "revol_util_clean",
    "total_acc", "credit_history_years", "collections_12_mths_ex_med", "acc_now_delinq",
    "tot_coll_amt", "tot_cur_bal", "acc_open_past_24mths", "avg_cur_bal", "bc_open_to_buy",
    "bc_util", "mort_acc", "pub_rec_bankruptcies", "tax_liens", "total_bal_ex_mort",
    "total_bc_limit", "total_il_high_credit_limit", "application_type",
]


STRICT_GOOD_TRAINING_STATUSES = {"Fully Paid"}
STRICT_BAD_TRAINING_STATUSES = {"Charged Off"}
STRICT_TRAINING_STATUSES = STRICT_GOOD_TRAINING_STATUSES | STRICT_BAD_TRAINING_STATUSES
NON_TERMINAL_EXCLUDED_STATUSES = {"Current", "In Grace Period", "Issued", "Late (31-120 days)", "Late (16-30 days)", "Default"}

TRACEABILITY_COLS = ["id", "loan_status", "issue_d", "issue_d_dt", "target_bad", "target_definition"]
REQUIRED_RAW_COLS = sorted(set(IDENTIFIER_COLS + TARGET_COLS + APPLICATION_TIME_CANDIDATES + LEAKAGE_DEMO_COLS))

print("Application-time candidates:", len(APPLICATION_TIME_CANDIDATES))
print("Safe starter features:", len(SAFE_STARTER_FEATURES))
print("Required raw columns before availability check:", len(REQUIRED_RAW_COLS))


## 3. Source Schema And Full Status Counts

**Description:** Read the raw header and count `loan_status` on the full accepted-loan file.

**Importance:** The cleaning sample should respect the imbalanced outcome distribution, but also keep enough bad/outlier statuses for QA.

**Process Made:** The notebook reads the header, intersects requested columns with available columns, and scans `loan_status` in chunks.

**Expected Results:** Available column list, unavailable requested columns, and full status counts.

**Caveats:** Full status counts are source-population counts. Final binary training counts change after unresolved statuses are removed.


In [ ]:
def read_header(path: Path) -> list[str]:
    opener = gzip.open if path.name.endswith(".gz") else open
    with opener(path, "rt", newline="", errors="replace") as f:
        return f.readline().rstrip("\n").split(",")


def save_table(df: pd.DataFrame, name: str, index: bool = False) -> Path:
    path = TABLE_DIR / f"{name}.csv"
    df.to_csv(path, index=index)
    print("Saved:", path)
    return path


accepted_columns = read_header(ACCEPTED_PATH)
ACCEPTED_USECOLS = [c for c in REQUIRED_RAW_COLS if c in accepted_columns]
MISSING_REQUESTED_COLS = sorted(set(REQUIRED_RAW_COLS) - set(ACCEPTED_USECOLS))
EXCLUDED_FROM_CLEANING_LOAD = [c for c in accepted_columns if c not in ACCEPTED_USECOLS]

status_counts = pd.Series(dtype="int64")
for chunk in pd.read_csv(ACCEPTED_PATH, usecols=["loan_status"], chunksize=CHUNK_SIZE, low_memory=False):
    status = chunk["loan_status"].astype("string").str.strip().fillna("Missing")
    status_counts = status_counts.add(status.value_counts(dropna=False), fill_value=0).astype("int64")

status_counts = status_counts.sort_values(ascending=False)
status_summary = status_counts.rename_axis("loan_status").reset_index(name="full_rows")
status_summary["full_pct"] = (status_summary["full_rows"] / status_summary["full_rows"].sum() * 100).round(4)

print("Raw accepted columns:", len(accepted_columns))
print("Columns loaded for cleaning:", len(ACCEPTED_USECOLS))
print("Requested columns unavailable:", MISSING_REQUESTED_COLS)
display(status_summary)
save_table(status_summary, "cleaning_full_loan_status_counts")


## 4. Stratified Cleaning Sample Plan

**Description:** Build a training-status sampling plan by `loan_status` for local cleaning and QA.

**Importance:** The baseline training dataset should include only the two stable terminal outcomes: `Fully Paid` and `Charged Off`. `Default` has only 40 rows in the full file, so it is excluded from the baseline to avoid unstable learning from a tiny class.

**Process Made:** The notebook filters full status counts to strict terminal training statuses, starts from proportional counts, applies a minimum target for rare statuses when possible, and caps targets at available rows.

**Expected Results:** A sample plan that excludes non-training statuses and preserves the `Fully Paid` versus `Charged Off` imbalance.

**Caveats:** Stratified sampling is for cleaning QA and development. Final training/evaluation should use only the approved baseline statuses and a train/test split that preserves temporal validation. `Default` can be revisited only as a sensitivity analysis.


In [ ]:
def build_status_sample_plan(
    counts: pd.Series,
    target_rows: int,
    min_per_status: int,
    eligible_statuses: set[str],
) -> pd.DataFrame:
    counts = counts[counts.index.astype("string").isin(eligible_statuses)].astype(int)
    total = int(counts.sum())
    if total == 0:
        raise ValueError("No eligible terminal training statuses were found in status_counts.")

    target_rows = min(target_rows, total)
    plan = pd.DataFrame({"loan_status": counts.index, "full_rows": counts.values})
    plan["proportional_target"] = (plan["full_rows"] / total * target_rows).astype(int)
    plan["sample_target"] = plan["proportional_target"].clip(lower=1)

    for idx, row in plan.iterrows():
        if row["full_rows"] > 0 and row["sample_target"] < min_per_status:
            plan.loc[idx, "sample_target"] = min(min_per_status, int(row["full_rows"]))

    # Reduce only rows above their protected floor so rare statuses like Default remain visible.
    protected_floor = plan.apply(
        lambda row: min(min_per_status, int(row["full_rows"])) if row["full_rows"] > 0 else 0,
        axis=1,
    )
    while int(plan["sample_target"].sum()) > target_rows:
        reducible = plan["sample_target"] > protected_floor
        if not reducible.any():
            reducible = plan["sample_target"] > 1
        idx = plan.loc[reducible, "sample_target"].idxmax()
        plan.loc[idx, "sample_target"] -= 1

    while int(plan["sample_target"].sum()) < target_rows:
        room = plan["sample_target"] < plan["full_rows"]
        if not room.any():
            break
        idx = ((plan.loc[room, "full_rows"] - plan.loc[room, "sample_target"])).idxmax()
        plan.loc[idx, "sample_target"] += 1

    plan["sample_pct_of_status"] = (plan["sample_target"] / plan["full_rows"] * 100).round(4)
    plan["sample_share_pct"] = (plan["sample_target"] / plan["sample_target"].sum() * 100).round(4)
    return plan.sort_values("full_rows", ascending=False).reset_index(drop=True)


if RUN_FULL_CLEANING:
    eligible_status_counts = status_counts[status_counts.index.astype("string").isin(STRICT_TRAINING_STATUSES)]
    sample_plan = pd.DataFrame({
        "loan_status": eligible_status_counts.index,
        "full_rows": eligible_status_counts.values,
        "sample_target": eligible_status_counts.values,
        "sample_pct_of_status": 100.0,
        "sample_share_pct": (eligible_status_counts.values / eligible_status_counts.sum() * 100).round(4),
    })
else:
    sample_plan = build_status_sample_plan(
        status_counts,
        target_rows=CLEANING_SAMPLE_ROWS,
        min_per_status=MIN_ROWS_PER_STATUS_IN_SAMPLE,
        eligible_statuses=STRICT_TRAINING_STATUSES,
    )

display(sample_plan)
save_table(sample_plan, "cleaning_status_sample_plan")
print("Strict terminal training statuses:", sorted(STRICT_TRAINING_STATUSES))
print("Planned rows:", int(sample_plan["sample_target"].sum()))


## 5. Load Cleaning Data

**Description:** Load either the full strict-terminal accepted-loan source or the stratified cleaning sample.

**Importance:** Cleaning must be able to run locally for development and also scale to full strict-terminal data treatment when the project is ready.

**Process Made:** The sample loader reads chunks, keeps only statuses in the training sample plan, applies per-status sampling probabilities, and trims each status to its planned target.

**Expected Results:** A raw cleaning dataframe containing only reviewed source columns and strict terminal training statuses needed for treatment, target creation, traceability, and leakage audit.

**Caveats:** The sample loader excludes non-training statuses. Do not use sample-based counts as final population counts.


In [ ]:
def load_full_cleaning_frame(path: Path, usecols: list[str], eligible_statuses: set[str]) -> pd.DataFrame:
    parts = []
    for chunk in pd.read_csv(path, usecols=usecols, chunksize=CHUNK_SIZE, low_memory=False):
        status = chunk["loan_status"].astype("string").str.strip()
        kept = chunk.loc[status.isin(eligible_statuses)].copy()
        if len(kept):
            parts.append(kept)
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=usecols)


def load_stratified_cleaning_sample(
    path: Path,
    usecols: list[str],
    plan: pd.DataFrame,
    chunk_size: int,
    random_state: int,
) -> pd.DataFrame:
    target_by_status = dict(zip(plan["loan_status"].astype("string"), plan["sample_target"].astype(int)))
    eligible_statuses = set(target_by_status)
    local_rng = random.Random(random_state)
    reservoir_by_status: dict[str, pd.DataFrame] = {}

    for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunk_size, low_memory=False):
        status = chunk["loan_status"].astype("string").str.strip()
        eligible_mask = status.isin(eligible_statuses)
        if not eligible_mask.any():
            continue

        eligible_chunk = chunk.loc[eligible_mask].copy()
        eligible_chunk["_status_for_sampling"] = status.loc[eligible_mask].astype("string").values
        eligible_chunk["_sampling_key"] = [local_rng.random() for _ in range(len(eligible_chunk))]

        for status_value, group in eligible_chunk.groupby("_status_for_sampling", observed=True):
            status_key = str(status_value)
            target_n = target_by_status[status_key]
            previous = reservoir_by_status.get(status_key)
            combined = group if previous is None else pd.concat([previous, group], ignore_index=True)
            if len(combined) > target_n:
                combined = combined.nsmallest(target_n, "_sampling_key")
            reservoir_by_status[status_key] = combined

    if not reservoir_by_status:
        return pd.DataFrame(columns=usecols)

    sampled = pd.concat(reservoir_by_status.values(), ignore_index=True)
    sampled = sampled.sample(frac=1, random_state=random_state).reset_index(drop=True)
    sampled = sampled.drop(columns=["_status_for_sampling", "_sampling_key"], errors="ignore")
    return sampled


if RUN_FULL_CLEANING:
    accepted_raw = load_full_cleaning_frame(ACCEPTED_PATH, ACCEPTED_USECOLS, STRICT_TRAINING_STATUSES)
else:
    accepted_raw = load_stratified_cleaning_sample(
        ACCEPTED_PATH,
        ACCEPTED_USECOLS,
        sample_plan,
        chunk_size=CHUNK_SIZE,
        random_state=RANDOM_STATE,
    )

print("Raw cleaning frame shape:", accepted_raw.shape)
loaded_status_counts = accepted_raw["loan_status"].astype("string").str.strip().value_counts(dropna=False).rename_axis("loan_status").reset_index(name="loaded_rows")
loaded_plan_check = sample_plan[["loan_status", "sample_target"]].merge(loaded_status_counts, on="loan_status", how="left")
loaded_plan_check["loaded_rows"] = loaded_plan_check["loaded_rows"].fillna(0).astype(int)
loaded_plan_check["row_delta"] = loaded_plan_check["loaded_rows"] - loaded_plan_check["sample_target"]
if (loaded_plan_check["row_delta"] != 0).any():
    raise AssertionError(f"Loaded sample counts do not match plan:\n{loaded_plan_check}")
print("Loaded status counts:")
display(loaded_plan_check)
display(accepted_raw.head())


## 6. Type Conversion And Clean Helper Fields

**Description:** Convert raw LendingClub strings, dates, percentages, FICO ranges, and numeric fields into analysis-ready helper fields.

**Importance:** Model-ready data needs stable numeric and categorical representations. Raw formats such as `36 months`, `10+ years`, `13.99%`, and `Jan-2015` cannot be used directly without controlled parsing.

**Process Made:** The notebook preserves raw fields and adds cleaned helpers such as `term_months`, `emp_length_years`, `int_rate_clean`, `revol_util_clean`, `fico_mean`, date fields, and `credit_history_years`.

**Expected Results:** A treated dataframe with raw audit fields plus cleaned modeling helpers.

**Caveats:** The numeric conversion list is hardcoded by design. This prevents automatic conversion of identifiers, target fields, categorical codes, or leakage fields without analyst review.


In [ ]:
def parse_percent(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    cleaned = series.astype("string").str.replace("%", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")


def parse_term_months(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series.astype("string").str.extract(r"(\d+)")[0], errors="coerce").astype("Int16")


def parse_emp_length(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.lower().str.strip()
    out = pd.Series(pd.NA, index=series.index, dtype="Float64")
    out[s.str.contains("< 1", na=False)] = 0
    out[s.str.contains(r"10\+", na=False)] = 10
    extracted = pd.to_numeric(s.str.extract(r"(\d+)")[0], errors="coerce")
    out = out.fillna(extracted)
    return out


def parse_lc_month(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, format="%b-%Y", errors="coerce")


NUMERIC_CANDIDATES = [
    # Hardcoded by design: these are fields expected to be numeric after EDA cleaning.
    # Avoid automatic numeric inference because LendingClub contains identifiers,
    # target fields, categorical codes, leakage fields, and sparse structural fields
    # that require explicit review before modeling.
    "loan_amnt", "installment", "annual_inc", "dti", "delinq_2yrs", "inq_last_6mths",
    "mths_since_last_delinq", "mths_since_last_record", "open_acc", "pub_rec", "revol_bal",
    "total_acc", "collections_12_mths_ex_med", "mths_since_last_major_derog", "acc_now_delinq",
    "tot_coll_amt", "tot_cur_bal", "annual_inc_joint", "dti_joint", "total_rev_hi_lim",
    "acc_open_past_24mths", "avg_cur_bal", "bc_open_to_buy", "bc_util", "mort_acc",
    "pub_rec_bankruptcies", "tax_liens", "tot_hi_cred_lim", "total_bal_ex_mort", "total_bc_limit",
    "total_il_high_credit_limit", "out_prncp", "total_pymnt", "total_rec_prncp", "total_rec_int",
    "recoveries", "collection_recovery_fee", "last_pymnt_amnt", "last_fico_range_high", "last_fico_range_low",
    "open_acc_6m", "open_act_il", "open_il_12m", "open_il_24m", "mths_since_rcnt_il",
    "total_bal_il", "il_util", "open_rv_12m", "open_rv_24m", "max_bal_bc", "all_util",
    "inq_fi", "total_cu_tl", "inq_last_12m", "chargeoff_within_12_mths", "delinq_amnt",
    "mo_sin_old_il_acct", "mo_sin_old_rev_tl_op", "mo_sin_rcnt_rev_tl_op", "mo_sin_rcnt_tl",
    "mths_since_recent_bc", "mths_since_recent_bc_dlq", "mths_since_recent_inq",
    "mths_since_recent_revol_delinq", "num_accts_ever_120_pd", "num_actv_bc_tl",
    "num_actv_rev_tl", "num_bc_sats", "num_bc_tl", "num_il_tl", "num_op_rev_tl",
    "num_rev_accts", "num_rev_tl_bal_gt_0", "num_sats", "num_tl_120dpd_2m", "num_tl_30dpd",
    "num_tl_90g_dpd_24m", "num_tl_op_past_12m", "pct_tl_nvr_dlq", "percent_bc_gt_75",
]

CATEGORICAL_CANDIDATES = [
    "grade", "sub_grade", "emp_length", "home_ownership", "verification_status", "loan_status",
    "purpose", "zip_code", "addr_state", "initial_list_status", "application_type",
    "verification_status_joint", "disbursement_method", "hardship_flag", "debt_settlement_flag",
]


def add_clean_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in ["int_rate", "revol_util"]:
        if col in out:
            out[f"{col}_clean"] = parse_percent(out[col])

    if "term" in out:
        out["term_months"] = parse_term_months(out["term"])
    if "emp_length" in out:
        out["emp_length_years"] = parse_emp_length(out["emp_length"])
    if {"fico_range_low", "fico_range_high"}.issubset(out.columns):
        out["fico_mean"] = out[["fico_range_low", "fico_range_high"]].mean(axis=1)

    for col in ["issue_d", "earliest_cr_line", "last_pymnt_d", "last_credit_pull_d"]:
        if col in out:
            out[f"{col}_dt"] = parse_lc_month(out[col])

    if "issue_d_dt" in out:
        out["issue_year"] = out["issue_d_dt"].dt.year.astype("Int16")
        out["issue_quarter"] = out["issue_d_dt"].dt.to_period("Q").astype("string")
        out["issue_month"] = out["issue_d_dt"].dt.to_period("M").astype("string")

    if {"issue_d_dt", "earliest_cr_line_dt"}.issubset(out.columns):
        out["credit_history_years"] = (out["issue_d_dt"] - out["earliest_cr_line_dt"]).dt.days / 365.25

    for col in NUMERIC_CANDIDATES:
        if col in out:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    for col in CATEGORICAL_CANDIDATES:
        if col in out:
            out[col] = out[col].astype("string").str.strip()

    return out


# Visual pre-treatment check: show the analyst-curated conversion sets before any treatment.
raw_numeric_candidates_present = [c for c in NUMERIC_CANDIDATES if c in accepted_raw.columns]
raw_categorical_candidates_present = [c for c in CATEGORICAL_CANDIDATES if c in accepted_raw.columns]
raw_conversion_set_summary = pd.DataFrame([
    {"set_name": "NUMERIC_CANDIDATES", "columns_present": len(raw_numeric_candidates_present), "columns": ", ".join(raw_numeric_candidates_present)},
    {"set_name": "CATEGORICAL_CANDIDATES", "columns_present": len(raw_categorical_candidates_present), "columns": ", ".join(raw_categorical_candidates_present)},
])
raw_conversion_column_audit = pd.DataFrame([
    {
        "column": col,
        "planned_treatment": "numeric_coerce",
        "raw_dtype": str(accepted_raw[col].dtype),
        "raw_missing_pct": round(accepted_raw[col].isna().mean() * 100, 4),
        "sample_values": "; ".join(accepted_raw[col].dropna().astype("string").head(5).tolist()),
    }
    for col in raw_numeric_candidates_present
] + [
    {
        "column": col,
        "planned_treatment": "categorical_string_strip",
        "raw_dtype": str(accepted_raw[col].dtype),
        "raw_missing_pct": round(accepted_raw[col].isna().mean() * 100, 4),
        "sample_values": "; ".join(accepted_raw[col].dropna().astype("string").head(5).tolist()),
    }
    for col in raw_categorical_candidates_present
])

print("Pre-treatment conversion set summary:")
display(raw_conversion_set_summary)
print("Pre-treatment conversion column audit:")
display(raw_conversion_column_audit)
save_table(raw_conversion_column_audit, "cleaning_pre_treatment_conversion_column_audit")

accepted_clean = add_clean_features(accepted_raw)
print("Cleaned frame shape:", accepted_clean.shape)
display(accepted_clean.head())


## 6.1 Type Conversion QA And Assertions

**Description:** Validate that Section 6 conversions actually produced the expected cleaned fields and dtypes.

**Importance:** Cleaning logic should fail loudly if a parser breaks, a source column disappears, or a supposedly numeric/categorical field remains in the wrong representation.

**Process Made:** The notebook checks expected helper columns, numeric dtypes, categorical string dtypes, valid term values, FICO mean consistency, date parsing, and non-negative credit-history years.

**Expected Results:** Assertion pass message plus exported QA tables for conversion status and derived-field quality.

**Caveats:** These checks validate format and basic business rules. They do not approve a field for modeling; leakage, timing, missingness, and compliance controls still apply later.


In [ ]:
EXPECTED_DERIVED_FIELDS = [
    "int_rate_clean", "revol_util_clean", "term_months", "emp_length_years", "fico_mean",
    "issue_d_dt", "earliest_cr_line_dt", "issue_year", "issue_quarter", "issue_month",
    "credit_history_years",
]
EXPECTED_DATE_FIELDS = [c for c in ["issue_d_dt", "earliest_cr_line_dt", "last_pymnt_d_dt", "last_credit_pull_d_dt"] if c in accepted_clean]
EXPECTED_NUMERIC_AFTER_CONVERSION = [c for c in raw_numeric_candidates_present + [
    "int_rate_clean", "revol_util_clean", "term_months", "emp_length_years", "fico_mean", "credit_history_years",
] if c in accepted_clean]
EXPECTED_CATEGORICAL_AFTER_CONVERSION = [c for c in raw_categorical_candidates_present if c in accepted_clean]

conversion_assertion_rows = []

def record_check(check_name: str, passed: bool, detail: str = "") -> None:
    conversion_assertion_rows.append({"check_name": check_name, "passed": bool(passed), "detail": detail})
    assert passed, f"{check_name} failed: {detail}"

missing_derived = [c for c in EXPECTED_DERIVED_FIELDS if c not in accepted_clean.columns]
record_check("expected_derived_fields_present", not missing_derived, f"missing={missing_derived}")

non_numeric = [c for c in EXPECTED_NUMERIC_AFTER_CONVERSION if not pd.api.types.is_numeric_dtype(accepted_clean[c])]
record_check("numeric_fields_are_numeric_dtype", not non_numeric, f"non_numeric={non_numeric}")

non_string_cats = [c for c in EXPECTED_CATEGORICAL_AFTER_CONVERSION if not pd.api.types.is_string_dtype(accepted_clean[c])]
record_check("categorical_fields_are_string_dtype", not non_string_cats, f"non_string={non_string_cats}")

non_datetime = [c for c in EXPECTED_DATE_FIELDS if not pd.api.types.is_datetime64_any_dtype(accepted_clean[c])]
record_check("date_fields_are_datetime_dtype", not non_datetime, f"non_datetime={non_datetime}")

if "term_months" in accepted_clean:
    invalid_terms = sorted(set(accepted_clean["term_months"].dropna().astype(int).unique()) - {36, 60})
    record_check("term_months_values_are_36_or_60", not invalid_terms, f"invalid_terms={invalid_terms}")

if {"fico_range_low", "fico_range_high", "fico_mean"}.issubset(accepted_clean.columns):
    expected_fico_mean = accepted_clean[["fico_range_low", "fico_range_high"]].mean(axis=1)
    fico_diff = (accepted_clean["fico_mean"] - expected_fico_mean).abs().dropna()
    record_check("fico_mean_matches_bounds", bool((fico_diff <= 1e-9).all()), f"max_diff={fico_diff.max() if len(fico_diff) else 0}")

if "credit_history_years" in accepted_clean:
    min_credit_history = accepted_clean["credit_history_years"].dropna().min()
    record_check("credit_history_years_non_negative", bool(pd.isna(min_credit_history) or min_credit_history >= 0), f"min={min_credit_history}")

conversion_assertions = pd.DataFrame(conversion_assertion_rows)
conversion_dtype_audit = pd.DataFrame([
    {
        "column": col,
        "expected_type_group": "numeric",
        "actual_dtype": str(accepted_clean[col].dtype),
        "missing_pct": round(accepted_clean[col].isna().mean() * 100, 4),
        "non_null": int(accepted_clean[col].notna().sum()),
    }
    for col in EXPECTED_NUMERIC_AFTER_CONVERSION
] + [
    {
        "column": col,
        "expected_type_group": "categorical_string",
        "actual_dtype": str(accepted_clean[col].dtype),
        "missing_pct": round(accepted_clean[col].isna().mean() * 100, 4),
        "non_null": int(accepted_clean[col].notna().sum()),
    }
    for col in EXPECTED_CATEGORICAL_AFTER_CONVERSION
] + [
    {
        "column": col,
        "expected_type_group": "datetime",
        "actual_dtype": str(accepted_clean[col].dtype),
        "missing_pct": round(accepted_clean[col].isna().mean() * 100, 4),
        "non_null": int(accepted_clean[col].notna().sum()),
    }
    for col in EXPECTED_DATE_FIELDS
])

derived_field_quality = pd.DataFrame([
    {
        "field": "term_months",
        "quality_check": "allowed_values",
        "result": sorted(accepted_clean["term_months"].dropna().astype(int).unique().tolist()) if "term_months" in accepted_clean else "missing",
    },
    {
        "field": "emp_length_years",
        "quality_check": "min_max",
        "result": f"{accepted_clean['emp_length_years'].min()} to {accepted_clean['emp_length_years'].max()}" if "emp_length_years" in accepted_clean else "missing",
    },
    {
        "field": "int_rate_clean",
        "quality_check": "min_max",
        "result": f"{accepted_clean['int_rate_clean'].min()} to {accepted_clean['int_rate_clean'].max()}" if "int_rate_clean" in accepted_clean else "missing",
    },
    {
        "field": "revol_util_clean",
        "quality_check": "min_max",
        "result": f"{accepted_clean['revol_util_clean'].min()} to {accepted_clean['revol_util_clean'].max()}" if "revol_util_clean" in accepted_clean else "missing",
    },
    {
        "field": "credit_history_years",
        "quality_check": "min_max",
        "result": f"{accepted_clean['credit_history_years'].min()} to {accepted_clean['credit_history_years'].max()}" if "credit_history_years" in accepted_clean else "missing",
    },
])

save_table(conversion_assertions, "cleaning_type_conversion_assertions")
save_table(conversion_dtype_audit, "cleaning_type_conversion_dtype_audit")
save_table(derived_field_quality, "cleaning_derived_field_quality")

print("Type conversion assertions passed:", len(conversion_assertions))
display(conversion_assertions)
print("Post-treatment dtype audit:")
display(conversion_dtype_audit)
print("Derived field quality:")
display(derived_field_quality)


## 7. Missing Labels, Duplicates, And Target Definition

**Description:** Standardize text missing labels, remove exact duplicate rows, and create the binary target.

**Importance:** Missing labels must be recognized as nulls, duplicates should not overweight loans, and target construction must avoid treating unresolved loans as good.

**Process Made:** The notebook converts conservative missing-label tokens to `pd.NA`, removes exact duplicates, maps strict terminal training statuses to `target_bad`, and labels non-terminal statuses as excluded from training.

**Expected Results:** A cleaned dataframe with `target_bad` and `target_definition`.

**Caveats:** `Current`, `In Grace Period`, `Late (16-30 days)`, `Late (31-120 days)`, `Default`, policy-exception statuses, and missing statuses are not used in this strict two-status baseline. They are excluded from binary training rather than treated as good or bad.


In [ ]:
STANDARD_MISSING_LABELS = {"", "N/A", "NA", "NO DATA", "NULL", "NAN", "MISSING", "UNKNOWN"}


def standardize_text_missing_labels(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = df.copy()
    rows = []
    for col in out.select_dtypes(include=["object", "string"]).columns:
        original_missing = int(out[col].isna().sum())
        stripped = out[col].astype("string").str.strip()
        token_mask = stripped.notna() & stripped.str.upper().isin(STANDARD_MISSING_LABELS)
        out[col] = stripped.mask(token_mask, pd.NA)
        rows.append({
            "column": col,
            "original_missing_count": original_missing,
            "standardized_missing_label_count": int(token_mask.sum()),
            "missing_count_after_standardization": int(out[col].isna().sum()),
            "n_unique_after_standardization": int(out[col].nunique(dropna=True)),
        })
    return out, pd.DataFrame(rows).sort_values("standardized_missing_label_count", ascending=False)


def add_target(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    status = out["loan_status"].astype("string").str.strip()

    # Strict terminal-outcome training definition.
    # Current, Grace Period, and Late statuses are snapshot states, not final repayment outcomes.
    good_terminal = STRICT_GOOD_TRAINING_STATUSES
    bad_terminal = STRICT_BAD_TRAINING_STATUSES
    non_terminal_excluded = NON_TERMINAL_EXCLUDED_STATUSES

    out["target_bad"] = pd.Series(pd.NA, index=out.index, dtype="Float64")
    out.loc[status.isin(good_terminal), "target_bad"] = 0.0
    out.loc[status.isin(bad_terminal), "target_bad"] = 1.0

    out["target_definition"] = "missing_or_review"
    out.loc[status.isin(good_terminal), "target_definition"] = "good_terminal_training"
    out.loc[status.isin(bad_terminal), "target_definition"] = "bad_terminal_training"
    out.loc[status.isin(non_terminal_excluded), "target_definition"] = "excluded_non_terminal_status"
    out.loc[~status.isin(good_terminal | bad_terminal | non_terminal_excluded), "target_definition"] = "excluded_not_in_strict_training_statuses"
    return out


accepted_clean, missing_label_summary = standardize_text_missing_labels(accepted_clean)
rows_before_dupes = len(accepted_clean)
duplicate_count = int(accepted_clean.duplicated().sum())
accepted_clean = accepted_clean.drop_duplicates().reset_index(drop=True)
accepted_clean = add_target(accepted_clean)

duplicate_summary = pd.DataFrame([{
    "rows_before": rows_before_dupes,
    "exact_duplicate_rows": duplicate_count,
    "rows_after": len(accepted_clean),
    "pct_removed": round(duplicate_count / rows_before_dupes * 100, 4) if rows_before_dupes else None,
}])

target_summary = (
    accepted_clean.groupby(["loan_status", "target_definition"], dropna=False, observed=True)["target_bad"]
    .agg(rows="size", bad_rows="sum")
    .reset_index()
)
target_summary["row_pct"] = (target_summary["rows"] / len(accepted_clean) * 100).round(4)

save_table(missing_label_summary, "cleaning_missing_label_standardization")
save_table(duplicate_summary, "cleaning_duplicate_summary")
save_table(target_summary, "cleaning_target_summary")

display(duplicate_summary)
display(target_summary)


## 8. Create Modeling Frame

**Description:** Remove non-terminal outcomes, identifiers, target columns, and leakage columns from the feature matrix.

**Importance:** This is the core treatment step that prevents outcome leakage and keeps the model population aligned with terminal accepted-loan repayment outcomes.

**Process Made:** The notebook keeps only rows with terminal target labels, writes traceability fields separately, and creates a starter model matrix from the approved 38 features.

**Expected Results:** `X_starter`, `y`, and `traceability` dataframes ready for modeling-prep splitting, imputation, encoding, and training.

**Caveats:** This notebook does not fit imputers or encoders. Those must be fit later on the training split only to avoid validation leakage.


In [ ]:
completed = accepted_clean[accepted_clean["target_bad"].notna()].copy().reset_index(drop=True)
completed["target_bad"] = completed["target_bad"].astype(int)

available_starter_features = [c for c in SAFE_STARTER_FEATURES if c in completed.columns]
missing_starter_features = sorted(set(SAFE_STARTER_FEATURES) - set(available_starter_features))

leakage_or_forbidden = set(IDENTIFIER_COLS + TARGET_COLS + LEAKAGE_DEMO_COLS + CLEAR_LEAKAGE_EXCLUDE + ["target_bad", "target_definition"])
forbidden_in_starter = sorted(set(available_starter_features) & leakage_or_forbidden)
if forbidden_in_starter:
    raise ValueError(f"Starter feature list contains forbidden fields: {forbidden_in_starter}")

X_starter = completed[available_starter_features].copy()
y = completed[["target_bad"]].copy()
traceability = completed[[c for c in TRACEABILITY_COLS if c in completed.columns]].copy()

modeling_frame_summary = pd.DataFrame([
    {"item": "raw_cleaning_rows", "value": len(accepted_raw)},
    {"item": "rows_after_duplicate_removal_and_targeting", "value": len(accepted_clean)},
    {"item": "completed_modeling_rows", "value": len(completed)},
    {"item": "starter_feature_count", "value": len(available_starter_features)},
    {"item": "missing_starter_features", "value": "; ".join(missing_starter_features)},
    {"item": "bad_rate", "value": round(float(y["target_bad"].mean()), 6) if len(y) else None},
])

missingness_model = pd.DataFrame({
    "feature": X_starter.columns,
    "missing_count": X_starter.isna().sum().values,
    "missing_pct": (X_starter.isna().mean().values * 100).round(4),
    "dtype": X_starter.dtypes.astype(str).values,
}).sort_values("missing_pct", ascending=False)

save_table(modeling_frame_summary, "cleaning_modeling_frame_summary")
save_table(missingness_model, "cleaning_starter_feature_missingness")

display(modeling_frame_summary)
display(missingness_model.head(40))
print("X_starter shape:", X_starter.shape)
print("y shape:", y.shape)
print("Traceability shape:", traceability.shape)


## 9. Export Treated Outputs

**Description:** Save cleaned artifacts for the modeling notebook.

**Importance:** Separating features, target, and traceability prevents accidental use of identifiers or outcome fields as predictors.

**Process Made:** The notebook writes parquet when available and falls back to CSV if the environment lacks a parquet engine.

**Expected Results:** Treated datasets under `Cleaning/cleaning_outputs/datasets/` and QA tables under `Cleaning/cleaning_outputs/tables/`.

**Caveats:** The exported starter matrix is cleaned but not imputed, encoded, scaled, or split. Modeling code must handle those steps inside a train-only pipeline.


In [ ]:
def save_dataset(df: pd.DataFrame, stem: str) -> Path:
    parquet_path = DATASET_DIR / f"{stem}.parquet"
    csv_path = DATASET_DIR / f"{stem}.csv"
    try:
        df.to_parquet(parquet_path, index=False)
        print("Saved:", parquet_path)
        return parquet_path
    except Exception as exc:
        print(f"Parquet save failed for {stem}: {exc}. Falling back to CSV.")
        df.to_csv(csv_path, index=False)
        print("Saved:", csv_path)
        return csv_path


export_paths = pd.DataFrame([
    {"artifact": "cleaned_completed_frame", "path": str(save_dataset(completed, "accepted_cleaned_completed_frame"))},
    {"artifact": "starter_feature_matrix", "path": str(save_dataset(X_starter, "accepted_X_starter"))},
    {"artifact": "target", "path": str(save_dataset(y, "accepted_y_target_bad"))},
    {"artifact": "traceability", "path": str(save_dataset(traceability, "accepted_traceability"))},
])
save_table(export_paths, "cleaning_export_paths")
display(export_paths)


## 10. Cleaning Conclusions

The cleaning notebook produces a leakage-clean starter modeling frame for accepted LendingClub loans.

Key conclusions:

1. The supervised baseline modeling population is accepted/originated loans with two stable terminal outcomes: `Fully Paid` and `Charged Off`.
2. `Current`, `In Grace Period`, `Late (16-30 days)`, `Late (31-120 days)`, `Default`, policy-exception statuses, and missing statuses are excluded from binary training in this strict two-status baseline.
3. Identifiers, target fields, and post-origination leakage fields are not allowed in the model matrix.
4. Clean helper fields are preferred over raw strings and date/range fields.
5. Missingness is preserved for downstream model-native handling or train-only imputation.
6. The starter feature matrix is ready for chronological split, train-only preprocessing, model fitting, calibration, and explainability analysis.
